# 02. LangChain RAG 구축
**Day 5**

> Day 4 **03 RAG** 에서 직접 짠 `split_text` · `search` 를  
> LangChain **Document → Splitter → Embedding → VectorStore** 로 바꿉니다.

**데이터:** `../4일차_찐/samples/pdf_samples/학칙.pdf` (이후 03에서 전체 PDF 확장)

---
## 0. 설치

In [ ]:
!pip install PyMuPDF pypdf langchain langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 48.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 18.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [langchain_community]ngchain_community]


---
## 1. Day 4 RAG vs LangChain RAG

| Day 4 (직접 구현) | LangChain |
|------------------|-----------|
| `pdf_to_text` | `PyMuPDFLoader` |
| `split_text` | `RecursiveCharacterTextSplitter` |
| `tokenize` + 교집합 | `OpenAIEmbeddings` + 벡터 유사도 |
| `INDEX` 리스트 | `FAISS` VectorStore |

---
## 2. Document 로드

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
import os 
# PDF 파일을 읽어서 텍스트 데이터를 추출합니다.
pdf_path = os.path.join(os.getcwd(),'samples/rag_samples/학칙.pdf')
loader = PyPDFLoader(pdf_path)

/var/folders/pz/g1s7yv693t5ddnqs0qzvsqvr0000gn/T/ipykernel_89650/1660011071.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
loader

In [ ]:
data_nyc = loader.load() #page content에 원문있음
print(data_nyc)

[Document(metadata={'producer': 'Hancom PDF 1.3.0.550', 'creator': 'Hwp 2024 13.0.0.3457', 'creationdate': '2026-06-17T16:37:51+09:00', 'author': 'DR', 'moddate': '2026-06-17T16:37:51+09:00', 'pdfversion': '1.4', 'source': '/Users/seorincho/Desktop/code/2026_AI/SK Autonomous R&D/5일차/samples/rag_samples/학칙.pdf', 'total_pages': 35, 'page': 0, 'page_label': '1'}, page_content='Ⅰ. 대학원 학칙  /  7\nⅠ. 대학원 학칙제정: 1974. 05. 18.개정(제122차): 2026. 06. 06.제1장 총칙제1조(목적) 대학원은 기독교 정신을 바탕으로 하여 창의적 이론과 과학적 방법을 탐구하고, 지도적 인격을 도야하여 인류 문화 향상에 기여함을 목적으로 한다.제2장 과정 및 정원제2조(과정) 대학원에는 석사학위를 취득하기 위한 석사학위과정, 박사학위를 취득하기 위한 박사학위과정, 석사학위과정을 거치지 않고 박사학위를 취득하기 위하여 석사학위과정과 박사학위과정이 통합된 과정(석·박사 통합과정, 이하 “통합과정”)을 두며, 학위과정 외에 학위를 수여하지 아니하는 연구과정을 둘 수 있다.제2조의2(협동과정) ① 대학원에 두는 학위과정으로 학과 외에 두 개 이상의 학과가 공동으로 설치·운영하는 협동과정(이하 “학과 간 협동과정”이라 한다)과 연구기관 또는 산업체와의 계약에 의하여 설치·운영하는 학·연·산 협동과정(이하 “학·연·산 협동과정”이라 한다)을 둘 수 있다.② 대학원장은 학과 간 협동과정의 운영실적을 2년마다 평가하여 대학원운영위원회 의결을 거쳐 존속여부를 결정하며, 이에 관한 사항은 따로 정한다.③ <삭제> 제2조의3 <삭제>제2조의4 <삭제>제2조의5(연계과정)

---
## 3. Text Split (청킹)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 데이터를 1000자 단위로 나눕니다. overlap은 100자로 설정합니다.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

In [ ]:
all_splits = text_splitter.split_documents(data_nyc)

In [ ]:
print(type(all_splits[0]))

<class 'langchain_core.documents.base.Document'>


In [ ]:
all_splits

[Document(metadata={'producer': 'Hancom PDF 1.3.0.550', 'creator': 'Hwp 2024 13.0.0.3457', 'creationdate': '2026-06-17T16:37:51+09:00', 'author': 'DR', 'moddate': '2026-06-17T16:37:51+09:00', 'pdfversion': '1.4', 'source': '/Users/seorincho/Desktop/code/2026_AI/SK Autonomous R&D/5일차/samples/rag_samples/학칙.pdf', 'total_pages': 35, 'page': 0, 'page_label': '1'}, page_content='Ⅰ. 대학원 학칙  /  7\nⅠ. 대학원 학칙제정: 1974. 05. 18.개정(제122차): 2026. 06. 06.제1장 총칙제1조(목적) 대학원은 기독교 정신을 바탕으로 하여 창의적 이론과 과학적 방법을 탐구하고, 지도적 인격을 도야하여 인류 문화 향상에 기여함을 목적으로 한다.제2장 과정 및 정원제2조(과정) 대학원에는 석사학위를 취득하기 위한 석사학위과정, 박사학위를 취득하기 위한 박사학위과정, 석사학위과정을 거치지 않고 박사학위를 취득하기 위하여 석사학위과정과 박사학위과정이 통합된 과정(석·박사 통합과정, 이하 “통합과정”)을 두며, 학위과정 외에 학위를 수여하지 아니하는 연구과정을 둘 수 있다.제2조의2(협동과정) ① 대학원에 두는 학위과정으로 학과 외에 두 개 이상의 학과가 공동으로 설치·운영하는 협동과정(이하 “학과 간 협동과정”이라 한다)과 연구기관 또는 산업체와의 계약에 의하여 설치·운영하는 학·연·산 협동과정(이하 “학·연·산 협동과정”이라 한다)을 둘 수 있다.② 대학원장은 학과 간 협동과정의 운영실적을 2년마다 평가하여 대학원운영위원회 의결을 거쳐 존속여부를 결정하며, 이에 관한 사항은 따로 정한다.③ <삭제> 제2조의3 <삭제>제2조의4 <삭제>제2조의5(연계과정)

In [ ]:
for i, split in enumerate(all_splits):
    print(f"Split {i+1}:------------------------------------\n")
    print(split)

Split 1:------------------------------------

page_content='Ⅰ. 대학원 학칙  /  7
Ⅰ. 대학원 학칙제정: 1974. 05. 18.개정(제122차): 2026. 06. 06.제1장 총칙제1조(목적) 대학원은 기독교 정신을 바탕으로 하여 창의적 이론과 과학적 방법을 탐구하고, 지도적 인격을 도야하여 인류 문화 향상에 기여함을 목적으로 한다.제2장 과정 및 정원제2조(과정) 대학원에는 석사학위를 취득하기 위한 석사학위과정, 박사학위를 취득하기 위한 박사학위과정, 석사학위과정을 거치지 않고 박사학위를 취득하기 위하여 석사학위과정과 박사학위과정이 통합된 과정(석·박사 통합과정, 이하 “통합과정”)을 두며, 학위과정 외에 학위를 수여하지 아니하는 연구과정을 둘 수 있다.제2조의2(협동과정) ① 대학원에 두는 학위과정으로 학과 외에 두 개 이상의 학과가 공동으로 설치·운영하는 협동과정(이하 “학과 간 협동과정”이라 한다)과 연구기관 또는 산업체와의 계약에 의하여 설치·운영하는 학·연·산 협동과정(이하 “학·연·산 협동과정”이라 한다)을 둘 수 있다.② 대학원장은 학과 간 협동과정의 운영실적을 2년마다 평가하여 대학원운영위원회 의결을 거쳐 존속여부를 결정하며, 이에 관한 사항은 따로 정한다.③ <삭제> 제2조의3 <삭제>제2조의4 <삭제>제2조의5(연계과정) 대학원에 두는 학위과정으로 본교 학부와 대학원을 연계하는 학부-대학원 연계과정을 둔다. 연계과정 운영에 관한 사항은 따로 정한다. 제2조의6(계약학과) ① 대학원에 두는 학위과정으로 국가, 지방자치단체 또는 산업체 등과의 계약에 의한 학과를 둘 수 있으며, 이의 운영에 관한 사항은 따로 정한다.② 대학원장은 대학원운영위원회 의결을 거쳐 계약학과의 존속 여부 등을 결정하며, 이에 관한 사항은 따로 정한다.' metadata={'producer': 'Hancom PDF 1.3.0.550', 'creator': 'Hwp 2024 13.0.0.3457', 'creationd

In [ ]:
print(all_splits[50].page_content)
print('----------------------')
print(all_splits[51].page_content)

Ⅰ. 대학원 학칙  /  31
석사학위과정박사학위과정대학학과대학학과
공과대학
항공우주공학과
공과대학
항공우주공학과기후변화에너지융합기술협동과정(모집종료) 기후변화에너지융합기술협동과정기술정책협동과정기술정책협동과정
계약학과
국방융합공학협동과정(폐과존치)
계약학과
국방융합공학협동과정(폐과존치)융합기술경영공학협동과정(폐과존치)반도체데이터사이언스학과이차전지융합공학과이차전지융합공학과우주국방융합학과우주국방융합학과디스플레이융합공학과디스플레이융합공학과시스템반도체공학과시스템반도체공학과디지털융합엔지니어링학과지능형데이터·최적화학과AI인베스트먼트공학과AI인베스트먼트공학과인공위성시스템학과인공위성시스템학과생명시스템대학생명과학부시스템생물학생명시스템대학생명과학부시스템생물학생화학생화학생명공학생명공학융합오믹스·의생명과학협동과정융합오믹스·의생명과학협동과정바이오산업공학협동과정바이오산업공학협동과정인공지능융합대학컴퓨터과학과인공지능융합대학컴퓨터과학과인공지능학과인공지능학과디지털애널리틱스융합협동과정계약학과모빌리티시스템융합학과지능융합학과신과대학신학과신과대학신학과
사회과학대학정치학과사회과학대학정치학과행정학과행정학과사회학과사회학과문화인류학과문화인류학과언론홍보영상학과언론홍보영상학과지역학협동과정지역학협동과정통일학협동과정통일학협동과정사회복지정책협동과정(구)법과대학법학과(구)법과대학법학과음악대학음악학과음악대학음악학과
----------------------
32
석사학위과정박사학위과정대학학과대학학과생활과학대학의류환경학과생활과학대학의류환경학과식품영양학과식품영양학과실내건축학과실내건축학과아동·가족학과아동·가족학과통합디자인학과통합디자인학과교육과학대학교육학과교육과학대학교육학과체육학과체육학과스포츠응용산업학과스포츠응용산업학과
의과대학
의학과
의과대학
의학과의과학과의과학과보건학과보건학과융합의학과융합의학과의료기기산업학과의료기기산업학과의생명시스템정보학과의생명시스템정보학과임상신약개발학과임상신약개발학과생체공학협동과정(폐과존치) 생체공학협동과정(폐과존치)언어병리학협동과정의료법·윤리학협동과정의료법·윤리학협동과정의학전산통계학협동과정의학전산통계학협

news.pdf 동일하게 진행

변수명:

loader_news, data_news, news_splits

In [ ]:
pdf_path = os.path.join(os.getcwd(),'samples/rag_samples/news.pdf')
loader_news = PyPDFLoader(pdf_path)
data_news = loader_news.load()
news_splits = text_splitter.split_documents(data_news)

In [ ]:
for i, split in enumerate(news_splits):
    print(f"Split {i+1}:------------------------------------")
    print(split)

Split 1:------------------------------------
page_content='1 
 
  
 
 
 
[2026 년 4 월 23 일] 
 
SK 하이닉스 2026 년 1 분기 경영실적 발표 
➢ 매출 52조 5,763억 원, 영업이익 37조 6,103억 원, 순이익 40조 3,459억 원 
➢ AI 수요 강세 속 고부가 제품 판매 확대… 사상 최대 분기 실적 달성 
➢ 에이전틱 AI 시대, D램∙낸드 전방위 수요 증가… 신제품으로 선제 대응 계획  
➢ “수요 가시성 고려한 투자로 공급 안정성∙재무 건전성 동시 확보할 것” 
 
SK하이닉스가 올해 1분기 매출액 52조 5,763억 원, 영업이익 37조 6,103억 원(영
업이익률 72%), 순이익 40조 3,459억 원(순이익률 77%)의 경영실적을 기록했다
고 23일 밝혔다. (K-IFRS 기준) 
  
분기 기준으로 볼 때, 매출은 사상 최초로 50조 원을 돌파했으며, 영업이익과 
영업이익률 역시 각각 37.6조 원, 72% 로 창사 이래 최고 기록을 달성했다. 영업이
익은 전분기 대비 2배 수준으로 급증하며 수익성 개선세를 뚜렷이 보여줬다. 
  
* 2025년 4분기 매출 32조 8,267억 원, 영업이익 19조 1,696억 원 
SK하이닉스는 "1분기는 계절적 비수기임에도 AI 인프라 투자 확대로 수요 강세
가 이어진 가운데, HBM·고용량 서버용 D램 모듈·eSSD 등 고부가가치 제품 판매를 
확대하며 실적 상승세를 이어갔다"고 설명했다. 
  
이 같은 실적 호조에 힘입어 1분기 말 현금성 자산은 전분기 말 대비 19.4조 
원 늘어난 54.3조 원을 기록했다. 반면, 차입금은 2.9조 원 감소한 19.3조 원을 기
록하며 35조 원의 순현금을 달성했다.' metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-06-25T02:14:09+09

In [ ]:
all_splits.extend(news_splits)

In [ ]:
all_splits

[Document(metadata={'producer': 'Hancom PDF 1.3.0.550', 'creator': 'Hwp 2024 13.0.0.3457', 'creationdate': '2026-06-17T16:37:51+09:00', 'author': 'DR', 'moddate': '2026-06-17T16:37:51+09:00', 'pdfversion': '1.4', 'source': '/Users/seorincho/Desktop/code/2026_AI/SK Autonomous R&D/5일차/samples/rag_samples/학칙.pdf', 'total_pages': 35, 'page': 0, 'page_label': '1'}, page_content='Ⅰ. 대학원 학칙  /  7\nⅠ. 대학원 학칙제정: 1974. 05. 18.개정(제122차): 2026. 06. 06.제1장 총칙제1조(목적) 대학원은 기독교 정신을 바탕으로 하여 창의적 이론과 과학적 방법을 탐구하고, 지도적 인격을 도야하여 인류 문화 향상에 기여함을 목적으로 한다.제2장 과정 및 정원제2조(과정) 대학원에는 석사학위를 취득하기 위한 석사학위과정, 박사학위를 취득하기 위한 박사학위과정, 석사학위과정을 거치지 않고 박사학위를 취득하기 위하여 석사학위과정과 박사학위과정이 통합된 과정(석·박사 통합과정, 이하 “통합과정”)을 두며, 학위과정 외에 학위를 수여하지 아니하는 연구과정을 둘 수 있다.제2조의2(협동과정) ① 대학원에 두는 학위과정으로 학과 외에 두 개 이상의 학과가 공동으로 설치·운영하는 협동과정(이하 “학과 간 협동과정”이라 한다)과 연구기관 또는 산업체와의 계약에 의하여 설치·운영하는 학·연·산 협동과정(이하 “학·연·산 협동과정”이라 한다)을 둘 수 있다.② 대학원장은 학과 간 협동과정의 운영실적을 2년마다 평가하여 대학원운영위원회 의결을 거쳐 존속여부를 결정하며, 이에 관한 사항은 따로 정한다.③ <삭제> 제2조의3 <삭제>제2조의4 <삭제>제2조의5(연계과정)

---
## 4. Embedding 

In [ ]:
!pip install langchain_chroma langchain_openai

  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.6/22.6 MB 31.0 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 30.5 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 27.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 28.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 32.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.1/721.1 kB 20.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 29.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 748.7/748.7 kB 23.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 30.4 MB/s  0:00:00m0:00:01
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39/39 [langchain_chroma][chromadb]s]-hub]p-proto-grpc]


In [ ]:
from langchain_openai import OpenAIEmbeddings 
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

In [ ]:
embedding = OpenAIEmbeddings(model='text-embedding-3-large', api_key=OPENAI_API_KEY)
#한국어는 3-large가 좋데. 

In [ ]:
v = embedding.embed_query('연세대학교에서 논문제출 시한을 연장한 학생은 휴학할 수 있는가? ')

In [ ]:
len(v)

3072

In [ ]:
v

[-0.019256591796875,
 -0.0037517547607421875,
 -0.025909423828125,
 0.035675048828125,
 0.04376220703125,
 -0.0229339599609375,
 0.005489349365234375,
 0.0210418701171875,
 0.0164794921875,
 -0.0279541015625,
 0.02117919921875,
 -0.0164031982421875,
 -0.00357818603515625,
 0.0005879402160644531,
 0.0253143310546875,
 -0.0204620361328125,
 -0.020416259765625,
 -0.0007710456848144531,
 -0.007144927978515625,
 0.006900787353515625,
 -0.022247314453125,
 -0.04156494140625,
 -0.0158843994140625,
 -0.00919342041015625,
 0.01812744140625,
 0.01971435546875,
 -0.0088958740234375,
 0.00144195556640625,
 -0.0215301513671875,
 -0.01531982421875,
 0.0019702911376953125,
 0.0103302001953125,
 0.0300750732421875,
 -0.0263671875,
 -0.015350341796875,
 -0.032928466796875,
 0.056060791015625,
 -0.0154571533203125,
 0.021881103515625,
 -0.027557373046875,
 -0.030303955078125,
 -0.03729248046875,
 0.0052490234375,
 -0.004833221435546875,
 0.03253173828125,
 -0.04095458984375,
 -0.0121002197265625,
 -0.04

#### Embedding 모델

text-embedding-3-large : 높은 정확도 (*한글)

text-embedding-3-small : 비용 대비 성능이 뛰어남

---
## 4. 벡터 저장

다양한 저장소 존재 (FAISS, **Chroma**, Qdrant, ....)

In [ ]:
from langchain_chroma import Chroma
import os

persist_directory = 'chroma_store'	

# 저장된 크로마 DB가 없다면 새로 만들기
if not os.path.exists(persist_directory):
    print("Creating new Chroma store")
    vectorstore = Chroma.from_documents(
        documents=all_splits,
        embedding=embedding,
        persist_directory=persist_directory
    )

else:
    print("Loading existing Chroma store")
    vectorstore = Chroma(		
        persist_directory=persist_directory, 
        embedding_function=embedding
    )

Creating new Chroma store


In [ ]:
#FAISS 예시
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(all_splits, embeddings)

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k":3})

docs = retriever.invoke("연세대학교에서 논문제출 시한을 연장한 학생은 휴학할 수 있는가? 예외는 무엇인가?")

In [ ]:
docs

[Document(id='4b5dc7c8-66cd-4dc3-9fc7-200fd3e4f4b4', metadata={'author': 'DR', 'page_label': '9', 'total_pages': 35, 'creator': 'Hwp 2024 13.0.0.3457', 'pdfversion': '1.4', 'source': '/Users/seorincho/Desktop/code/2026_AI/SK Autonomous R&D/5일차/samples/rag_samples/학칙.pdf', 'producer': 'Hancom PDF 1.3.0.550', 'moddate': '2026-06-17T16:37:51+09:00', 'creationdate': '2026-06-17T16:37:51+09:00', 'page': 8}, page_content='제33조 <삭제> 제34조(심사판정) 논문심사 합격판정은 석사학위논문 심사위원 3분의 2 이상, 박사학위논문 심사위원 5분의 4 이상의 찬성으로 이루어진다.제35조(재심사) ① 학위논문 심사에 불합격 판정을 받은 학생은 적절한 수정 보완을 하여 재심사를 받을 수 있으며 이에 관한 사항은 따로 정한다.② <삭제>제36조(논문제출 시한) ① 석사학위과정 학생은 입학일로부터 4년 이내에, 박사학위과정 학생은 입학일로부터 7년 이내에, 통합과정 학생은 입학일로부터 8년 이내에 학위논문 심사에 합격하여 완성된 학위논문을 대학원에서 정한 절차에 따라 제출하여야 한다.② 위 제1항의 시한에는 등록학기만 산입하며, 휴학 및 제적기간은 산입하지 아니한다.③ 수료요건을 충족하고 합당한 사유가 있는 학생은 논문제출 시한 연장 사유서와 연구계획서를 제출하고 지도교수와 대학원장의 승인을 얻어 논문제출 시한을 2년 연장할 수 있다.④ 논문제출 시한 연장을 승인받은 기간에는 휴학할 수 없으나 다음 각호의 휴학은 예외로 처리할 수 있다.1. 입대휴학 2. 육아휴학3. 그 외에 대학원장이 인정하는 중대한 사유에 의한 일반휴학⑤ 「장애인 등에 대한 

In [ ]:
for d in docs:
    print(d)
    print('------')

page_content='제33조 <삭제> 제34조(심사판정) 논문심사 합격판정은 석사학위논문 심사위원 3분의 2 이상, 박사학위논문 심사위원 5분의 4 이상의 찬성으로 이루어진다.제35조(재심사) ① 학위논문 심사에 불합격 판정을 받은 학생은 적절한 수정 보완을 하여 재심사를 받을 수 있으며 이에 관한 사항은 따로 정한다.② <삭제>제36조(논문제출 시한) ① 석사학위과정 학생은 입학일로부터 4년 이내에, 박사학위과정 학생은 입학일로부터 7년 이내에, 통합과정 학생은 입학일로부터 8년 이내에 학위논문 심사에 합격하여 완성된 학위논문을 대학원에서 정한 절차에 따라 제출하여야 한다.② 위 제1항의 시한에는 등록학기만 산입하며, 휴학 및 제적기간은 산입하지 아니한다.③ 수료요건을 충족하고 합당한 사유가 있는 학생은 논문제출 시한 연장 사유서와 연구계획서를 제출하고 지도교수와 대학원장의 승인을 얻어 논문제출 시한을 2년 연장할 수 있다.④ 논문제출 시한 연장을 승인받은 기간에는 휴학할 수 없으나 다음 각호의 휴학은 예외로 처리할 수 있다.1. 입대휴학 2. 육아휴학3. 그 외에 대학원장이 인정하는 중대한 사유에 의한 일반휴학⑤ 「장애인 등에 대한 특수교육법」 제15조 제1항의 특수교육대상자는 「대학원 학칙」 제2조의8 제2항에 따라 합당한 사유가 있는 경우 대학원장의 승인을 받아 제3항의 연장 시한을 예외로 한다.제36조의2(논문제출) 학위논문 인준을 받은 학생은 대학원에서 정한 절차에 따라 완성된 학위논문을 제출하여야 한다.제8장 연구과정생, 외국인 학생, 공개강좌제37조(연구과정생) 연구과정생은 대학원 강의를 청강할 수 있으나 이수학점으로 인정되지 않으며 정해진 수업료를 납부하여야 하며, 이수자에게는 재학사실증명서를 발행할 수 있다.제38조(외국인 학생) 외국인으로서 본 대학원에 입학하고자 하는 자는 한국어 또는 영어 사용 능력이 일정한 수준에 도달하여야 하며, 별도의 입학전형에 합격할 경우 정원외로 입학을 허가할 수 있다. 단, 대학원에서 별도로 지정한 학과는

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

question = '석사학위과정 수업연한은?'
chat = ChatOpenAI(model="gpt-4o-mini")
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

In [ ]:
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x11a5a0ad0>, search_kwargs={'k': 3})

In [ ]:
rag_prompt = ChatPromptTemplate.from_template("""
사용자의 질문에 대해 아래 context에 기반하여 답변하라

문맥:
{context}

질문: {question}
""")

### rag 답변 생성해보기

In [ ]:
# 힌트
def format_docs(docs):
    return '\n\n'.join(d.page_content for d in docs)

In [ ]:
format_docs(retriever.invoke(question)) #왜 invoke에 question이 들어가는거야????????????????

'34\n[별표 2] 학위과정별, 학과 및 전공별 학위수여명1. 석사학위과정(신촌캠퍼스)대학학과국문학위명영문학위명후드코드\n문과대학\n국어국문학과문학석사M.A. in Korean Language and LiteraturePANTONE 5315C중어중문학과문학석사M.A. in Chinese Language and LiteraturePANTONE 5315C영어영문학과문학석사M.A. in English Language and LiteraturePANTONE 5315C독어독문학과문학석사M.A. in German Language and LiteraturePANTONE 5315C불어불문학과문학석사M.A. in French Language and LiteraturePANTONE 5315C노어노문학과문학석사M.A. in Russian Language and LiteraturePANTONE 5315C사학과문학석사M.A. in HistoryPANTONE 5315C철학과문학석사M.A. in PhilosophyPANTONE 5315C문헌정보학과문헌정보학석사M.A. in Library and Information SciencePANTONE 5315C기록관리학석사M.A. in Record Management and ArchivesPANTONE 5315C심리학과문학석사M.A. in PsychologyPANTONE 5315C심리학석사M.S. in PsychologyPANTONE 5315C인지과학석사M.S. in Cognitive SciencePANTONE 5315C\n협동과정\n\nⅠ. 대학원 학칙  /  29\n[별표 1]학위과정별(캠퍼스) 입학정원 현황(2026학년도)□ 일반 학위과정캠퍼스석사 정원(명)박사 정원(명)신촌1,3281)1,0292)국제12271합계1,4501,1001) 통계데이터사이언스학과(첨단분야 교원확보율 충족형) 석사 정원 8명 증원2) 통계데이터사이언스학과(첨단분야 교원확보율 충족형) 박사 정원 3명 증원□ 계약학과 학위과정 [정원 외]1. 신촌캠퍼스구분석사 정원

In [ ]:
chain = rag_prompt|chat
chain.invoke({'context':format_docs(retriever.invoke(question)),
'question':question})

AIMessage(content='석사학위과정의 수업연한에 대한 정보는 제공된 문맥에서 직접적으로 언급되어 있지 않습니다. 일반적으로 석사학위과정의 수업연한은 2년 정도로 설정되어 있지만, 각 대학과 프로그램에 따라 다를 수 있습니다. 따라서 정확한 수업연한은 해당 학과의 공식 자료를 확인하는 것이 좋습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 85, 'prompt_tokens': 1009, 'total_tokens': 1094, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_42e6cfce1b', 'id': 'chatcmpl-Duw4beBCQdPONC8saoaLHg4n1YwYJ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f0303-3df2-7733-bcbd-06a2b46baa8b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1009, 'output_tokens': 85, 'total_tokens': 1094, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': 

## Retriever -> Tool

In [ ]:
from langchain_core.tools import create_retriever_tool
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini")
search_tool = create_retriever_tool(
    retriever,
    name = 'search_pdf_documents',
    description = ' pdf samples 안의 문서함에서 질문과 관련된 내용을 검색합니다.'
)

In [ ]:
retriever

In [ ]:
retriever.invoke('석사 수업연한')

In [ ]:
search_tool.invoke('석사 수업연한')

In [ ]:
tools = [search_tool]
tool_dict = {'search_pdf_documents':search_tool}

llm_with_tools = llm.bind_tools(tools)

messages = [
    HumanMessage('석사학위과정 수업연한은?')
]

response = llm_with_tools.invoke('석사학위과정 수업연한은?')
messages.append(response)

In [ ]:
response.tool_calls

In [ ]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # (7) tool_dict를 사용하여 도구 함수를 선택
    print(tool_call["args"]) # (8) 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # (9) 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

In [ ]:
llm_with_tools.invoke(messages)